![Week 1 Day 2](../assets/banners/w1d2.png)

# Week 1 Day 2 - How LLMs Work, and Your First Real LangChain Calls

Welcome back! 👋

Yesterday you got your environment working, made your very first LLM call and had a model mark Sam's deadlock answer. It *mostly* worked, and you were (rightly) a little suspicious: would the mark come out the same every time? What does a call like that actually cost?

Today we open the bonnet. You'll see the three ideas every AI engineer reasons with all day long, **tokens**, **context windows** and **temperature**, and you'll turn them into real numbers for Asha's CS201 midterm. Then we tour the LangChain 1.x ecosystem, and put `init_chat_model` through its paces: invoke, stream, batch, and swapping models without touching your code.

![Your 6-week roadmap](../assets/roadmap/week1.png)

### 🎬 Videos for this lab
| # | Lecture |
|:-:|---|
| 1 | Day 2 - How LLMs Work: Tokens, Context Windows, Temperature and Cost |
| 2 | Day 2 - The LangChain 1.x Ecosystem: Core, LangGraph, LangSmith, Deep Agents |
| 3 | Day 2 - First Calls with `init_chat_model`: Invoke, Stream, Batch, Swap Providers |

### 🔁 Recap from Day 1
- `init_chat_model(settings.chat_model)` gives you a chat model; `.invoke()` returns an `AIMessage`
- System messages set the rules, user messages make the request
- A one-prompt grader is a fun demo, but not something to trust with 200 scripts

### 🎯 By the end of this lab you will
1. 🔢 Count tokens yourself with `tiktoken`, and estimate what the whole exam pack costs to send
2. 🌡️ See exactly what temperature does to a model's next-token choice
3. 🗺️ Know which part of the LangChain ecosystem does what
4. ⚡ Invoke, stream and batch a chat model, and swap models from `.env` or at run time

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Most cells in Parts 1 to 3 run <b>offline</b> and cost nothing. Only Part 4 calls a real model, and the whole lab costs a fraction of a cent. If a cell shows a red <i>Missing API key</i> panel, go back to Part 5 of <code>setup/SETUP-PC.md</code> (or <code>-mac</code> / <code>-linux</code>): you can still run every offline cell.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

import numpy as np
import tiktoken
from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise.config import DATA_DIR, settings

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: Tokens, the unit models read, write and bill in

A model never sees letters or words. Before your text reaches it, a **tokenizer** chops it into **tokens**: common words become one token, rarer words get split into pieces, and spaces and punctuation are tokens too. On average one token is about ¾ of an English word, or roughly 4 characters.

Why care? Because **everything is measured in tokens**: the price you pay (per million input and output tokens), how much text fits in a request (the context window), and how long a reply takes (output tokens are generated one at a time).

`tiktoken` is OpenAI's tokenizer. `o200k_base` is the encoding used by OpenAI's recent models. Other providers use different tokenizers, so treat these counts as a good estimate rather than an exact bill.

In [ ]:
encoding = tiktoken.get_encoding("o200k_base")

question = "What is a deadlock? Name TWO of the four necessary conditions for a deadlock. [4 marks]"
tokens = encoding.encode(question)

print(f"{len(question)} characters → {len(tokens)} tokens\n")
# 💡 Show each token between | bars, so you can see where the tokenizer cut
print("|".join(encoding.decode([t]) for t in tokens))

In [ ]:
# 💡 Spaces attach to the START of the next word, and even "deadlock" is two pieces.
#    Common words are cheap; jargon, typos and unusual names get split further.
for word in ["deadlock", "semaphore", "Coffman", "preemption", "deadlok", "Northbridge", "TCP", "2^19"]:
    pieces = [encoding.decode([t]) for t in encoding.encode(word)]
    print(f"{word:<12} {len(pieces)} token(s): {pieces}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Asha's exam pack is a 2-page question paper, an 8-page marking scheme and a 3-page methodology. Before you run the next cell, guess: how many tokens is the whole pack? 2,000? 10,000? 50,000? And how many tokens is the JSON we want back?</span>
        </td>
    </tr>
</table>

In [ ]:
from gradewise.exam.parse import parse_pdf  # 💡 PDF → clean Markdown; we study it properly on Day 4 and Day 5

exam_dir = DATA_DIR / "exams" / "cs_midterm"
pack_tokens = 0
for name in ["question_paper.pdf", "marking_scheme.pdf", "methodology.pdf"]:
    doc = parse_pdf(exam_dir / name)
    n = len(encoding.encode(doc.text))
    pack_tokens += n
    print(f"{name:<22} {doc.page_count:>2} pages  {len(doc.text):>6,} characters  {n:>6,} tokens")

expected_json = (exam_dir / "expected_exam.json").read_text(encoding="utf-8")
json_tokens = len(encoding.encode(expected_json))
print(f"\nThe whole pack: {pack_tokens:,} tokens in. The ExamPaper JSON we want: about {json_tokens:,} tokens out.")

### Context windows

A model's **context window** is the maximum number of tokens it can handle in one request: your instructions, the documents, the conversation so far **and** its reply, all together. Today's models have windows from about 128,000 to over 1,000,000 tokens.

The exam pack fits easily. But a context window is a *limit*, not a target:

| Myth | Reality |
|---|---|
| "If it fits, just send everything" | You pay for every input token, on every call |
| "The model reads all of it equally well" | Models attend less reliably to details buried in the middle of very long inputs |
| "Long outputs are fine" | Output tokens cost several times more than input tokens, and are generated one by one |

This is exactly why, in **week 2**, GradeWise won't paste all 40 pages of course notes into every grading call. It will *retrieve* the two or three paragraphs that matter: that's RAG.

In [ ]:
# 💡 A quick budget: what share of a few typical context windows does one extraction call use?
call_tokens = pack_tokens + json_tokens
for window in [128_000, 400_000, 1_000_000]:
    print(f"{window:>9,}-token window: one extraction call uses {call_tokens / window:6.1%}")

### Cost

Providers charge per **million tokens**, with a separate (higher) price for output. The prices below are **placeholders** in the right ballpark for a small, cheap model. Look up the real ones for your model on your provider's pricing page and paste them in.

In [ ]:
# ✋ TODO (optional): replace these with your model's real prices, in US dollars per million tokens
PRICE_PER_MILLION = {"input": 0.75, "output": 4.50}


def estimate_cost(input_tokens: int, output_tokens: int, prices: dict = PRICE_PER_MILLION) -> float:
    """Dollars for one call: input and output tokens are billed separately."""
    return (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000


one_call = estimate_cost(pack_tokens, json_tokens)
print(f"Extracting the exam once ≈ ${one_call:.4f}")
print(f"Asha's whole department (40 exams a term) ≈ ${40 * one_call:.2f}")
print(f"Share of the cost that is OUTPUT: {json_tokens * PRICE_PER_MILLION['output'] / 1e6 / one_call:.0%}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Being able to estimate cost <i>before</i> you build is a superpower in client conversations. "Turning an exam pack into JSON costs about five cents" is a sentence that gets a project approved. Notice that output tokens dominate: that's why GradeWise asks for compact, structured answers instead of chatty essays.</span>
        </td>
    </tr>
</table>

## Part 2: Temperature, or why Sam's mark kept changing

An LLM writes one token at a time. At each step it produces a **score** (a *logit*) for every token in its vocabulary, turns those scores into **probabilities** with a function called *softmax*, and then **samples** the next token.

**Temperature** divides the scores before softmax:
- **Low temperature (→ 0):** the most likely token wins almost every time. Predictable, repeatable.
- **Temperature 1:** the model's natural distribution.
- **High temperature (> 1):** the distribution flattens, so unlikely tokens get picked more often. Creative, but erratic.

Let's simulate it with made-up scores for the next word in *"A deadlock is when processes ___"*.

In [ ]:
candidates = ["wait", "block", "compete", "starve", "dance"]
logits = np.array([4.0, 3.2, 1.5, 1.0, -1.0])  # 💡 made-up scores: "wait" is the model's favourite


def softmax_with_temperature(scores: np.ndarray, temperature: float) -> np.ndarray:
    scaled = scores / max(temperature, 1e-6)  # ⚠️ temperature 0 would divide by zero, so we clamp it
    exp = np.exp(scaled - scaled.max())  # 💡 subtracting the max keeps exp() from overflowing
    return exp / exp.sum()


for t in [0.1, 0.5, 1.0, 2.0]:
    probs = softmax_with_temperature(logits, t)
    row = "  ".join(f"{word}={p:5.1%}" for word, p in zip(candidates, probs, strict=True))
    print(f"T={t:<4} {row}")

In [ ]:
# 💡 Now SAMPLE ten next words at each temperature, the way a model does when it writes
rng = np.random.default_rng(seed=7)  # 💡 a fixed seed, so you see the same "random" picks as the video

for t in [0.1, 1.0, 2.0]:
    picks = rng.choice(candidates, size=10, p=softmax_with_temperature(logits, t))
    print(f"T={t:<4}", " ".join(picks))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">For <b>extraction and grading</b> you want the same answer every time, so GradeWise asks for <code>temperature=0</code>. But here's a real-world gotcha: OpenAI's <b>reasoning</b> models (the gpt-5 family) don't accept a temperature at all, and <code>langchain-openai</code> quietly drops it. Run the next cell to see. With those models, consistency has to come from <b>structure and validation</b> instead: that's Day 3 and week 3.</span>
        </td>
    </tr>
</table>

In [ ]:
# 💡 Offline check: we build the model object with a dummy key. Nothing is sent anywhere.
if settings.provider == "openai":
    from langchain_openai import ChatOpenAI

    model_name = settings.chat_model.split(":", 1)[1]
    probe = ChatOpenAI(model=model_name, temperature=0, api_key="sk-not-a-real-key")
    print(f"Asked {model_name} for temperature=0 → the model object keeps temperature={probe.temperature}")
else:
    print(f"Your provider is {settings.provider}: temperature=0 is passed straight through.")

## Part 3: The LangChain 1.x ecosystem in one picture

"LangChain" is a family of packages, and it helps to know which one does what. Here's the map, and where each piece shows up in GradeWise:

![Diagram](../assets/diagrams/8adc811d58ce.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart TB
  subgraph Core["langchain-core · the shared vocabulary"]
    M[Messages & content blocks] --- R[Runnables: invoke · stream · batch] --- T[Tools] --- P[Prompt templates]
  end
  Prov["Provider packages<br/>langchain-openai · -anthropic · -google-genai · -ollama"] --> Core
  LC["langchain · init_chat_model, create_agent, middleware"] --> Core
  LG["LangGraph · stateful graphs, checkpoints, human-in-the-loop"] --> Core
  LC --> LG
  DA["Deep Agents · planning, sub-agents, files"] --> LG
  LS["LangSmith · tracing & evaluation"] -.observes.-> LC & LG
```

</details>


| Package | What it gives you | Where GradeWise uses it |
|---|---|---|
| `langchain-core` | Messages, Runnables, tools, prompts: the interfaces everything else speaks | Every week |
| `langchain-openai` (and friends) | The adapters that talk to each provider's API | Behind `gradewise.models` |
| `langchain` | `init_chat_model`, `create_agent`, middleware | Weeks 1, 3, 5 |
| `langgraph` | Graphs with state, retries, parallel steps, `interrupt` for human approval | Weeks 3 to 5 |
| `langsmith` | Traces of every call, datasets and evaluations | Week 5 |
| Deep Agents | A higher layer for long, multi-step agents | A look ahead in week 6 |

Let's check which versions you have installed:

In [ ]:
from importlib.metadata import PackageNotFoundError, version

for package in ["langchain-core", "langchain", "langgraph", "langchain-openai", "langchain-anthropic", "langsmith"]:
    try:
        print(f"{package:<22} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<22} (not installed)")

### The one interface to rule them all: Runnables

Chat models, prompts, retrievers, tools, even whole graphs share one interface, called a **Runnable**:

| Method | What it does | When to use it |
|---|---|---|
| `.invoke(x)` | one input → one output | most of the time |
| `.stream(x)` | yields the output in chunks as it's generated | chat UIs, long answers |
| `.batch([x1, x2, ...])` | many inputs, run in parallel | marking 30 answers at once |

The `a`-prefixed versions (`ainvoke`, `astream`, `abatch`) do the same with `async`. Learn these three once and you can drive anything in LangChain.

Here's a dress rehearsal with a **fake** model that ships with `langchain-core`: it replies with canned text and costs nothing. Watch `.stream()` deliver the reply in pieces:

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake = GenericFakeChatModel(messages=iter(["A deadlock is when every process waits for another, forever."]))

for chunk in fake.stream("What is a deadlock?"):
    print(chunk.content, end="|", flush=True)  # 💡 the | shows where each chunk ends

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">GradeWise's tests use fakes like this one so they run for free in seconds (open <code>tests/test_exam.py</code> to see). You'll use the same trick on Day 4 to rehearse a tool loop before spending a cent.</span>
        </td>
    </tr>
</table>

## Part 4: `init_chat_model` for real: invoke, stream, batch

Time to spend a fraction of a cent. We create the model exactly as on Day 1, from the `provider:model` string in your `.env`:

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(settings.chat_model)

response = llm.invoke("In two sentences: what is the difference between TCP and UDP?")
display(Markdown(response.content))
print(response.usage_metadata)

In [ ]:
# 💡 Compare with your own tiktoken count: the provider's is a little higher, because it wraps
#    your message in a few formatting tokens of its own.
my_count = len(encoding.encode("In two sentences: what is the difference between TCP and UDP?"))
usage = response.usage_metadata
print(f"tiktoken says {my_count} tokens; the provider billed {usage['input_tokens']} input tokens")
print(f"This call cost about ${estimate_cost(usage['input_tokens'], usage['output_tokens']):.6f}")

### Streaming

`.stream()` gives you an iterator of `AIMessageChunk`s. Printing each one as it arrives is what makes chat apps feel fast: the first words appear in well under a second, even when the full answer takes several.

In [ ]:
for chunk in llm.stream("Explain paging to a first-year student in about 80 words."):
    print(chunk.content, end="", flush=True)

### Batching

`.batch()` sends several inputs at once and returns the replies **in the same order**. `max_concurrency` caps how many run at the same time, which keeps you under your provider's rate limits.

In [ ]:
questions = [
    "In one sentence, what is a race condition?",
    "In one sentence, what does the A in ACID stand for, and what does it mean?",
    "In one sentence, what does DNS do?",
]

replies = llm.batch(questions, config={"max_concurrency": 3})

for q, reply in zip(questions, replies, strict=True):
    print(f"❓ {q}\n💬 {reply.content}\n")

### Swapping models

Because every model speaks the same Runnable interface, swapping is a configuration change, never a code change. There are two ways:

**1. In `.env` (what GradeWise does).** Change one line and restart the kernel. Some provider strings you could use (keys and details in `setup/SWAP_PROVIDER.md`):

| `CHAT_MODEL=` | Provider | Key needed |
|---|---|---|
| `openai:...` | OpenAI | `OPENAI_API_KEY` |
| `anthropic:...` | Anthropic (Claude) | `ANTHROPIC_API_KEY` |
| `google_genai:...` | Google (Gemini) | `GOOGLE_API_KEY` |
| `ollama:...` | A free local model via Ollama | none |

**2. At run time.** `init_chat_model` can create a *configurable* model, where you choose the model per call. Here we compare your everyday model with your smart (grading) model on the same question:

In [ ]:
switchable = init_chat_model(configurable_fields=("model",))

for model_name in dict.fromkeys([settings.chat_model, settings.smart_model]):  # 💡 de-duplicated, order kept
    reply = switchable.invoke(
        "In one sentence, why must a page size be a power of two?",
        config={"configurable": {"model": model_name}},
    )
    print(f"🧠 {model_name}\n   {reply.content}\n")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">From Day 3 onwards we stop calling <code>init_chat_model</code> ourselves and use <code>gradewise.models</code>: <code>chat_model()</code>, <code>grading_model()</code> and <code>vision_model()</code> read the three tiers from your <code>.env</code>, and <code>BUDGET_MODE=true</code> collapses them all onto the cheap model. Open <code>gradewise/models.py</code> and find <code>make_chat_model</code>: it's <code>init_chat_model</code> plus a friendly key check.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🔢 counted tokens with `tiktoken`, and found the whole exam pack is under 7,000 tokens in, and the JSON about 11,000 out
- 💰 turned tokens into dollars, and saw that output tokens dominate the bill
- 🌡️ watched temperature reshape the next-token distribution, and learned why reasoning models ignore it
- 🗺️ mapped the LangChain ecosystem: core, providers, `langchain`, LangGraph, LangSmith, Deep Agents
- ⚡ used `invoke`, `stream` and `batch`, and swapped models from `.env` and at run time

But every reply so far has been **free text**. Software can't reliably read a mark out of a paragraph. **Tomorrow (Day 3)** we fix that for good: prompt templates, Pydantic models and `with_structured_output`, so the model answers in exactly the shape we ask for, and corrects itself when it doesn't.

⏭ **Next:** `week1/day3.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>Batch on a budget.</b> Pick three questions from Asha's question paper. (1) Before calling anything, count the prompt tokens with <code>tiktoken</code> and estimate the cost with <i>your</i> model's real prices. (2) Use <code>llm.batch(...)</code> to ask for a short model answer to all three at once. (3) Compare your estimate with the <code>usage_metadata</code> you got back. How close were you? Bonus: stream the longest answer. Solution in <code>week1/solutions/day2_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangChain chat models: <a href="https://docs.langchain.com/oss/python/langchain/models">docs.langchain.com/oss/python/langchain/models</a><br/> • Try tokenizing in your browser: <a href="https://platform.openai.com/tokenizer">platform.openai.com/tokenizer</a><br/> • Model prices: <a href="https://openai.com/api/pricing">openai.com/api/pricing</a> and your provider's equivalent<br/> • <code>guides/09_ai_apis_costs_and_ollama</code> for the free local route</span>
        </td>
    </tr>
</table>